# Named Entity Recognition (NER) with Word Embeddings

**Goal:** teach a model to find *named entities* in English text: people (PER), organisations (ORG), locations (LOC) and other names (MISC, e.g. nationalities or events).

For example, in *"Paul Kagame met the president of Kenya in Nairobi."* we want: `Paul Kagame` = PER, `Kenya` = LOC, `Nairobi` = LOC.

**Plan:**
1. Load and explore the CoNLL-2003 dataset
2. Baseline model **without** embeddings (the "before")
3. Model **with GloVe** embeddings (the "after")
4. Model with **FastText** embeddings, and how each handles unknown words
5. Compare all three and look at mistakes
6. Test on a small set of Rwandan sentences
7. Try the best model on new sentences

## Step 1: Load the dataset
We import the libraries we need. `datasets` downloads data from Hugging Face, and `seqeval` scores NER predictions per entity type.

In [1]:
import warnings
warnings.filterwarnings("ignore")

from collections import Counter

import numpy as np
import pandas as pd
import datasets
from datasets import load_dataset

datasets.disable_progress_bars()  # keep the notebook output clean

We load **CoNLL-2003**, the standard English NER dataset (news articles from Reuters).

The original Hugging Face version (`conll2003`) uses an old "loading script" that newer versions of `datasets` refuse to run. So we load Hugging Face's automatic **parquet copy of the same dataset** (`eriktks/conll2003`, revision `refs/convert/parquet`). It has the same sentences, the same labels and the same official train / validation / test splits.

In [2]:
conll = load_dataset("eriktks/conll2003", revision="refs/convert/parquet")
conll

DatasetDict({
    train: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 14041
    })
    validation: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 3250
    })
    test: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 3453
    })
})

Each sentence is a list of **tokens** (words and punctuation) and a matching list of **NER tags**. The tags are stored as numbers, so we turn them back into their names (like `B-PER`).

In [3]:
TAG_NAMES = conll["train"].features["ner_tags"].feature.names
print("Tag set:", TAG_NAMES)

def to_sentences(split):
    """Return two lists: the tokens of each sentence and their tag names."""
    tokens = [list(t) for t in split["tokens"]]
    tags = [[TAG_NAMES[i] for i in seq] for seq in split["ner_tags"]]
    return tokens, tags

train_tokens, train_tags = to_sentences(conll["train"])
val_tokens, val_tags = to_sentences(conll["validation"])
test_tokens, test_tags = to_sentences(conll["test"])

for word, tag in zip(train_tokens[0], train_tags[0]):
    print(f"{word:10} {tag}")

Tag set: ['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC', 'B-MISC', 'I-MISC']


EU         B-ORG
rejects    O
German     B-MISC
call       O
to         O
boycott    O
British    B-MISC
lamb       O
.          O


### What do the tags mean? (the BIO format)
- `B-` means the **B**eginning of an entity, and `I-` means **I**nside it (the 2nd, 3rd... word of the same entity).
- `O` means **O**utside: the word is not part of any entity.

So "Paul Kagame" is tagged `B-PER I-PER`. There are 4 entity types: **PER** (person), **ORG** (organisation), **LOC** (location) and **MISC** (other names, like "German" or "World Cup").

**Note:** CoNLL-2003 has **no DATE type**. Words like "Friday" or "1996" are tagged `O`, so none of our models can learn to find dates.

### How big is each split?

In [4]:
def count_entities(tag_seqs):
    """Count entities by counting the B- tags (each entity starts with exactly one)."""
    counts = Counter(t[2:] for seq in tag_seqs for t in seq if t.startswith("B-"))
    return {etype: counts[etype] for etype in ["PER", "ORG", "LOC", "MISC"]}

rows = []
for name, toks, tags in [("train", train_tokens, train_tags),
                         ("validation", val_tokens, val_tags),
                         ("test", test_tokens, test_tags)]:
    row = {"split": name, "sentences": len(toks), "tokens": sum(len(s) for s in toks)}
    row.update(count_entities(tags))
    rows.append(row)

split_summary = pd.DataFrame(rows).set_index("split")
split_summary

,sentences,tokens,PER,ORG,LOC,MISC
split,,,,,,
train,14041,203621,6600,6321,7140,3438
validation,3250,51362,1842,1341,1837,922
test,3453,46435,1617,1661,1668,702


How common is each tag? Most tokens are `O`, so the dataset is very **imbalanced**. That's why we judge models with entity-level precision, recall and F1, not plain accuracy: a model that tags everything `O` would already have high accuracy.

In [5]:
tag_counts = Counter(t for seq in train_tags for t in seq)
tag_share = pd.DataFrame({"count": pd.Series(tag_counts)}).loc[TAG_NAMES]
tag_share["percent"] = (100 * tag_share["count"] / tag_share["count"].sum()).round(2)
tag_share

,count,percent
O,169578,83.28
B-PER,6600,3.24
I-PER,4528,2.22
B-ORG,6321,3.10
I-ORG,3704,1.82
B-LOC,7140,3.51
I-LOC,1157,0.57
B-MISC,3438,1.69
I-MISC,1155,0.57
